[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Early Stopping


## What you will be able to do

Let a validation set decide how many trees a LightGBM or XGBoost model gets, read which round it
chose, and make sure every prediction uses that round.

## The idea

### The problem

The **Why Gradient Boosting** notebook trained LightGBM with 300 trees. In its solutions, 50 trees
left a test loss of 2.758, 300 trees 2.542 and 1,000 trees 2.654, so somewhere between 300 and 1,000
more trees stopped helping. Picking the count from those test losses would make the test stations one
more thing the model was fitted to, and the number they report would stop measuring how well it does
on stations it has never seen.

The right number also moves. It depends on the learning rate, on how large each tree may grow and on
the data, so a count that suited one setting is wrong for the next. The **Learning Rate and Trees** notebook
avoided a run for every count by training 2,000 trees at each rate and reading the lowest point of the
validation loss recorded after every round. That still trains every tree up to a ceiling fixed in
advance, many of them past the best round, and leaves the reader to tell `predict` how many trees to
use.

### A formal definition

> In gradient boosting, **early stopping** trains with a large number of rounds as a ceiling and
> measures the loss on a **validation set** after every round. When the validation loss has not improved for a set number
> of rounds, the **patience**, training stops, and the model keeps the round at which the validation
> loss was lowest, its **best iteration**.

### Why it works that way

Boosting adds one tree per round, so a single run passes through every candidate count on its way to
the ceiling, and the validation loss after each round is the score that many trees would get. The
training loss keeps falling as trees are added. The validation loss falls, flattens and then drifts
up once the trees start fitting the training stations' noise, and early stopping reads that curve as
it is drawn.

The patience exists because the curve is not smooth. A loss that fails to improve for a few rounds
may improve again, so a run waits for a longer stretch, here usually 100 rounds, before deciding that
the lowest point is behind it.

The validation stations have now been used to choose something, so their loss at the best iteration
flatters the model a little, and the test stations are measured once, at the end. The two libraries
also keep different things. LightGBM drops the trees after the best iteration. XGBoost keeps them all
and records which round was best, and only its scikit-learn class reads that record when it predicts.

### Where this shows up

The **Tuning** notebook combines early stopping with cross-validation, and the **A Frost Forecast**
notebook stops its models this way. scikit-learn's `HistGradientBoostingRegressor` has an
`early_stopping` setting, and the **PyTorch and TensorFlow, Deep Dive** guide stops a network's
training on its validation loss in the same way.

### What this notebook covers

- the training and validation losses after every round, from one run
- early stopping in LightGBM with a callback, and in XGBoost with a setting
- the best iteration in each library, and what each keeps
- how the patience changes where a run stops
- the test stations, measured once
- four mistakes, two of which raise nothing

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
import lightgbm as lgb

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, (600, 1))
y = np.sin(x.ravel()) + rng.normal(0, 0.3, 600)

model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.1, verbose=-1)
model.fit(x[:400], y[:400], eval_set=[(x[400:], y[400:])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
print(model.best_iteration_, model.booster_.num_trees(), len(model.evals_result_["valid_0"]["l2"]))
```

```
45 45 95
```

The model was allowed 1,000 trees and trained 95. The validation loss on the last 200 points was
lowest after 45, it did not improve in the 50 rounds after that, and the model kept those 45 trees.

## Setup

The table, the two boosting libraries, and a score.

- `numpy` and `pandas` build the table
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `mean_squared_error` scores a model's predictions

`station_days` makes the same 20,000 made-up station-days as the **Why Gradient Boosting** notebook,
drawn from a generator started from a fixed seed, so the table is the same on every run. The cell
splits the stations into 300 for training, 50 for validation and 50 for testing, keeping all of a
station's days on one side, and then takes each part's eight features as `X_train`, `X_valid` and
`X_test` and its lowest temperatures as `y_train`, `y_valid` and `y_test`.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]
X_train, y_train = train[FEATURES], train["tmin"]
X_valid, y_valid = valid[FEATURES], valid["tmin"]
X_test, y_test = test[FEATURES], test["tmin"]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### The validation stations

In [2]:
for name, rows in (("training", train), ("validation", valid), ("test", test)):
    print(f"{name:10} {rows['station'].nunique():3} stations, {len(rows):6,} rows")

training   300 stations, 15,000 rows
validation  50 stations,  2,500 rows
test        50 stations,  2,500 rows


The 50 validation stations are the ones every model in this notebook is stopped on. None of their days
is in the training rows, so their loss behaves like the loss on any new station.

### Both losses after every round

`eval_set` takes a list of tables for LightGBM to score after every round, and `eval_names` names
them. The loss it records is `l2`, LightGBM's name for the mean squared error. The run below has no
early stopping, so it trains all 2,000 trees.

In [3]:
curve = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, random_state=0, verbose=-1)
curve.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
          eval_names=["training", "validation"])
history = curve.evals_result_

for trees in (50, 100, 200, 500, 1000, 2000):
    print(f"{trees:5} trees   training {history['training']['l2'][trees - 1]:.3f}"
          f"   validation {history['validation']['l2'][trees - 1]:.3f}")

validation_curve = np.array(history["validation"]["l2"])
print("lowest validation loss:", round(validation_curve.min(), 3), "after", validation_curve.argmin() + 1, "trees")

   50 trees   training 3.220   validation 4.021
  100 trees   training 1.819   validation 2.697
  200 trees   training 1.401   validation 2.501
  500 trees   training 1.082   validation 2.495
 1000 trees   training 0.788   validation 2.530
 2000 trees   training 0.476   validation 2.589
lowest validation loss: 2.48 after 361 trees


The training loss falls at every printed count, from 3.220 after 50 trees to 0.476 after 2,000. The
validation loss reaches its lowest, 2.48, after 361 trees and then rises slowly, to 2.589 by 2,000,
so the trees after the 361st fitted the training stations more closely and the validation stations
less well.

### Early stopping in LightGBM

LightGBM stops through a **callback**, a function `fit` calls after every round. `lgb.early_stopping`
makes one with a patience of 100, and `verbose=False` stops it printing a line when training starts
and the best iteration when it stops.

In [4]:
lightgbm_model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, random_state=0, verbose=-1)
lightgbm_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)],
                   callbacks=[lgb.early_stopping(100, verbose=False)])

print("best iteration:", lightgbm_model.best_iteration_)
print("validation loss there:", round(lightgbm_model.best_score_["valid_0"]["l2"], 3))
print("rounds trained:", len(lightgbm_model.evals_result_["valid_0"]["l2"]))
print("trees kept:", lightgbm_model.booster_.num_trees())

best iteration: 361
validation loss there: 2.48
rounds trained: 461
trees kept: 361


The run stopped after 461 rounds, 100 after the best, and kept 361 trees, the lowest point of the
curve above. LightGBM counts its best iteration as a number of trees, and the model now holds only
those, so every prediction uses them.

### Early stopping in XGBoost

XGBoost takes the patience as a setting, `early_stopping_rounds`, and the validation table as
`eval_set` in `fit`, where `verbose=False` stops it printing the loss after every round.

In [5]:
xgboost_model = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.05, early_stopping_rounds=100,
                                 random_state=0)
xgboost_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

print("best iteration:", xgboost_model.best_iteration)
print("best score:", round(xgboost_model.best_score, 3), "| squared:", round(xgboost_model.best_score ** 2, 3))
print("rounds kept:", xgboost_model.get_booster().num_boosted_rounds())
print("validation mean squared error:", round(mean_squared_error(y_valid, xgboost_model.predict(X_valid)), 3))

best iteration: 262
best score: 1.587 | squared: 2.517
rounds kept: 363
validation mean squared error: 2.517


XGBoost counts its best iteration in rounds from 0, so 262 means 263 trees. `best_score` is the
validation loss in the metric XGBoost watched, the root mean squared error, and its square is
2.517, the validation loss that `predict` gives. The booster still holds all 363 rounds it trained,
100 past the best, and `predict` uses only the first 263. XGBoost's best count differs from
LightGBM's 361. Their defaults grow trees of different sizes, which the **Tree Shape** notebook
measures, and XGBoost penalizes leaf values with a `lambda` of 1. The validation loss above is 2.501
after 200 trees, 2.48 after 361 and 2.495 after 500, so a small difference between two models can move
the best round a long way.

### How the patience changes where a run stops

In [6]:
for patience in (5, 20, 100, 300):
    model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, random_state=0, verbose=-1)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)],
              callbacks=[lgb.early_stopping(patience, verbose=False)])
    rounds = len(model.evals_result_["valid_0"]["l2"])
    print(f"patience {patience:3}: best {model.best_iteration_:3} trees, "
          f"validation {model.best_score_['valid_0']['l2']:.3f}, {rounds} rounds trained")

patience   5: best 155 trees, validation 2.523, 160 rounds trained
patience  20: best 179 trees, validation 2.501, 199 rounds trained
patience 100: best 361 trees, validation 2.480, 461 rounds trained
patience 300: best 361 trees, validation 2.480, 661 rounds trained


A patience of 5 stopped after 160 rounds, with a best of 155 trees and a validation loss of 2.523,
because the curve went five rounds without improving long before its lowest point. A patience of 20
got a little further, and 100 and 300 both found 361 trees, the second after training 200 more
rounds that found nothing lower.

### The test stations measured once

Both models were stopped on the validation stations, so the test stations can now say how well each
does on stations neither has seen.

In [7]:
print("LightGBM:", round(mean_squared_error(y_test, lightgbm_model.predict(X_test)), 3))
print("XGBoost:", round(mean_squared_error(y_test, xgboost_model.predict(X_test)), 3))

LightGBM: 2.538
XGBoost: 2.492


LightGBM's loss on the test stations is 2.538 and XGBoost's 2.492. Each number is a fair measure for
its model, because nothing was chosen from it, and the gap between them comes from one split and one
seed, too little to rank the libraries.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/05-early-stopping-solutions.ipynb).

**1.** Train XGBoost with a patience of 20 in place of 100, at a learning rate of 0.05, and print its best iteration and the number of trees its `predict` uses.

In [8]:
# your code here

**2.** Train LightGBM with early stopping at a learning rate of 0.1 and at 0.02, with a patience of 100 and room for 5,000 trees, and print each model's best iteration and validation loss.

In [9]:
# your code here

**3.** Pass `min_delta=0.01` to `lgb.early_stopping`, so that a round counts as an improvement only if it lowers the validation loss by at least 0.01, and print the best iteration and validation loss.

In [10]:
# your code here

**4.** Find, in `history`, the first round at which the training loss falls below 1.0, and print the validation loss at that round beside the lowest validation loss.

In [11]:
# your code here

**5.** Train XGBoost with `xgb.train`, the native interface, with `evals` and `early_stopping_rounds=100`, and show that `booster.predict` with `iteration_range=(0, booster.best_iteration + 1)` gives the same validation predictions as `xgboost_model.predict`.

In [12]:
# your code here

## Common errors

### `ValueError: For early stopping, at least one dataset and eval metric is required for evaluation`

In [13]:
lgb.LGBMRegressor(n_estimators=2000, verbose=-1).fit(X_train, y_train,
                                                    callbacks=[lgb.early_stopping(100, verbose=False)])

ValueError: For early stopping, at least one dataset and eval metric is required for evaluation

LightGBM's callback needs a validation table to score, and `fit` was given none. Pass the validation
stations as `eval_set=[(X_valid, y_valid)]`, as the worked examples do.

### `ValueError: Must have at least 1 validation dataset for early stopping.`

In [14]:
xgb.XGBRegressor(n_estimators=2000, early_stopping_rounds=100).fit(X_train, y_train, verbose=False)

ValueError: Must have at least 1 validation dataset for early stopping.

XGBoost's `early_stopping_rounds` has the same need, and the same `eval_set` is the fix.

### No error, and a booster that predicts with every tree

XGBoost's scikit-learn class hands its trees to a native `Booster`, which a program can take with
`get_booster()`, for instance to save it. The model below learns at 0.3, so its best round comes early.

In [15]:
fast = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.3, early_stopping_rounds=100, random_state=0)
fast.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
booster = fast.get_booster()
valid_matrix = xgb.DMatrix(X_valid, enable_categorical=True)

print("best iteration:", fast.best_iteration, "| rounds in the booster:", booster.num_boosted_rounds())
print("validation, fast.predict:   ", round(mean_squared_error(y_valid, fast.predict(X_valid)), 3))
print("validation, booster.predict:", round(mean_squared_error(y_valid, booster.predict(valid_matrix)), 3))

best iteration: 25 | rounds in the booster: 126
validation, fast.predict:    2.881
validation, booster.predict: 2.946


The booster holds all 126 rounds, 100 after the best one, and its `predict` uses every tree it holds,
which raises the validation loss from 2.881 to 2.946. Only the scikit-learn class reads the best
iteration when it predicts. LightGBM's booster has no such gap, since it kept only its best trees.

In [16]:
best_only = booster.predict(valid_matrix, iteration_range=(0, fast.best_iteration + 1))
print("validation, best trees only:", round(mean_squared_error(y_valid, best_only), 3))
print("same as fast.predict:", np.allclose(best_only, fast.predict(X_valid)))

validation, best trees only: 2.881
same as fast.predict: True


Given `iteration_range=(0, fast.best_iteration + 1)`, the booster uses the first 26 trees and
predicts what `fast.predict` does.

### No error, and a test loss chosen by the test stations

In [17]:
peeking = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, random_state=0, verbose=-1)
peeking.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(100, verbose=False)])
stopped = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, random_state=0, verbose=-1)
stopped.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])

print("stopped on the test stations:      best", peeking.best_iteration_,
      "| test", round(mean_squared_error(y_test, peeking.predict(X_test)), 3))
print("stopped on the validation stations: best", stopped.best_iteration_,
      "| test", round(mean_squared_error(y_test, stopped.predict(X_test)), 3))

stopped on the test stations:      best 233 | test 2.528
stopped on the validation stations: best 361 | test 2.538


Stopped on the test stations, the model scores 2.528 on them, below the 2.538 of the model stopped on
the validation stations. The test stations chose the stopping round, 233, as the round where their own
loss was lowest, so 2.528 is the best of the losses they were shown, and it no longer estimates how
the model does on stations it has never seen. Stop on the validation stations, and measure the test
stations once.

## Recap

- You can give LightGBM a validation table and an `early_stopping` callback, and XGBoost a validation
  table and `early_stopping_rounds`.
- You can read the best iteration in each, knowing that LightGBM counts trees from 1 and XGBoost
  rounds from 0, and that XGBoost's `best_score` is a root mean squared error.
- You can choose a patience long enough that a flat stretch does not end a run early.
- You can predict with only the best trees from an XGBoost booster, and keep the test stations out of
  every choice.

## What is next

The **Tree Shape** notebook looks at the trees themselves: how many leaves each library grows, how
deep they go, and the settings that limit both.

---

&#8592; **Previous:** [Learning Rate and Trees](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/04-learning-rate-and-trees.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Tree Shape](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/06-tree-shape.ipynb) &#8594;
